
# ============================================================
# 1. SETUP & DATA LOADING
# ============================================================

In [2]:
import pandas as pd
import numpy as np
import re
import html
import json
import warnings
import joblib
from bs4 import BeautifulSoup
import contractions
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

In [3]:
print("🔹 Loading data...")
df = pd.read_csv("dataset/train_data.csv")

print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")

🔹 Loading data...
Loaded 838944 rows and 11 columns.


In [4]:
# Basic info
print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
df.info()
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
print(df.describe())


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 838944 entries, 0 to 838943
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   overall         838944 non-null  int64 
 1   vote            191468 non-null  object
 2   verified        838944 non-null  bool  
 3   reviewTime      838944 non-null  object
 4   reviewerID      838944 non-null  object
 5   asin            838944 non-null  object
 6   style           490613 non-null  object
 7   reviewerName    838717 non-null  object
 8   reviewText      838944 non-null  object
 9   summary         838868 non-null  object
 10  unixReviewTime  838944 non-null  int64 
dtypes: bool(1), int64(2), object(8)
memory usage: 64.8+ MB

Missing values:
overall                0
vote              647476
verified               0
reviewTime             0
reviewerID             0
asin                   0
style             348331
reviewerName         227
review

In [5]:
# Quick checks
print(df.shape)
print(df["overall"].value_counts())
print(df["unixReviewTime"].min(), df["unixReviewTime"].max())

# Duplicates
print(
    "Duplicate rows:",
    df.duplicated(subset=["reviewText", "summary", "reviewerID", "asin"]).sum(),
)

# Empty text checks
print("Empty reviewText:", (df["reviewText"].str.strip() == "").sum())

# Label distribution grouped by asin and reviewer
print("Products with most reviews:", df["asin"].value_counts().head())
print("Top reviewers:", df["reviewerID"].value_counts().head())

(838944, 11)
overall
5    461485
4    156514
1     82950
3     81239
2     56756
Name: count, dtype: int64
1451606400 1538524800
Duplicate rows: 8459
Empty reviewText: 0
Products with most reviews: asin
B010OYASRG    1853
B00L0YLRUW    1205
B00P7EVST6    1115
B01DA0YCNC     996
B00S9SGNNS     905
Name: count, dtype: int64
Top reviewers: reviewerID
A680RUE1FDO8B     158
A1UQUDT2Q0YENM    132
A1RHJX6OA0O9KQ    128
A1EXGL6L0QQ0M5    127
AVU1ILDDYW301     115
Name: count, dtype: int64


# ------------------------------------------------------------
# 2. TEXT CLEANING FUNCTION
# ------------------------------------------------------------

In [6]:
nltk.download(["wordnet", "punkt", "omw-1.4"])
lemmatizer = WordNetLemmatizer()


def clean_text(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    try:
        s = str(s)
        s = html.unescape(s)
        s = BeautifulSoup(s, "lxml").get_text()
        s = s.lower()

        # URL + number cleanup
        s = re.sub(r"http\S+", " <URL> ", s)
        s = re.sub(r"www\S+", " <URL> ", s)
        s = re.sub(r"\b\d{4,}\b", " <NUM> ", s)
        s = re.sub(r"\b\d+\.\d+\b", " <NUM> ", s)

        # Expand contractions (e.g., don't -> do not)
        s = contractions.fix(s)

        # Remove special characters
        s = re.sub(r"[^a-z0-9\s\.\!\?]", " ", s)
        s = re.sub(r"\s+", " ", s).strip()

        # Lemmatize tokens
        tokens = s.split()
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        return " ".join(tokens)
    except Exception as e:
        print(f"Error cleaning text: {e}")
        return ""

[nltk_data] Downloading package wordnet to /home/aliqnbri/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/aliqnbri/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/aliqnbri/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# ------------------------------------------------------------
# 3. DATA CLEANING & MISSING VALUES
# ------------------------------------------------------------

In [7]:
print("🔹 Handling missing values and cleaning text...")

df["style"] = df["style"].fillna("unknown")
df["summary"] = df["summary"].fillna("")
df["reviewText"] = df["reviewText"].fillna("")

# Combine summary + review
df["text"] = (df["summary"] + ". " + df["reviewText"]).str.strip()

# Drop duplicates & empty text
before = len(df)
df = df.drop_duplicates(subset=["reviewText", "summary", "reviewerID", "asin"])
df = df[df["text"].str.strip() != ""].reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate/empty rows.")

# Clean text
df["text_clean"] = df["text"].apply(clean_text)

🔹 Handling missing values and cleaning text...


Removed 8459 duplicate/empty rows.


# ------------------------------------------------------------
# 4. FEATURE ENGINEERING
# ------------------------------------------------------------

In [8]:
print("🔹 Feature engineering...")


# Parse helpful votes
def parse_vote(x):
    try:
        return int(str(x).split()[0])
    except:
        return 0


df["vote_count"] = df["vote"].apply(parse_vote)
df["has_vote"] = (df["vote_count"] > 0).astype(int)
df["verified"] = df["verified"].astype(int)

# Text statistics
df["review_len_chars"] = df["reviewText"].str.len()
df["review_len_words"] = df["reviewText"].str.split().apply(len)
df["summary_len"] = df["summary"].str.split().apply(len)

# Time features
df["review_year"] = pd.to_datetime(df["reviewTime"]).dt.year
df["review_month"] = pd.to_datetime(df["reviewTime"]).dt.month

# Frequency encodings
asin_freq = df["asin"].value_counts().to_dict()
df["asin_freq"] = df["asin"].map(asin_freq)
style_freq = df["style"].value_counts().to_dict()
df["style_freq"] = df["style"].map(style_freq)

# Label encoding for style
le_style = LabelEncoder()
df["style_encoded"] = le_style.fit_transform(df["style"])

🔹 Feature engineering...


# ------------------------------------------------------------
# 5. OUTLIER HANDLING & FEATURE TRANSFORMATION
# ------------------------------------------------------------

In [9]:
print("🔹 Handling outliers and transforming skewed features...")

# Log-transform skewed numeric features
df['vote_count'] = np.log1p(df['vote_count'])
df['asin_freq'] = np.log1p(df['asin_freq'])
df['style_freq'] = np.log1p(df['style_freq'])

# IQR-based winsorization (clipping extreme values)
num_cols = ['vote_count', 'review_len_chars', 'review_len_words', 
            'summary_len', 'asin_freq', 'style_freq']

for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = np.clip(df[col], lower, upper)


🔹 Handling outliers and transforming skewed features...


# ------------------------------------------------------------
# 6. NORMALIZE FEATURES
# ------------------------------------------------------------

In [10]:
print("🔹 Normalizing numeric features...")
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
joblib.dump(scaler, "clean_dataset/scaler.pkl")
print("✅ Saved scaler.pkl")

🔹 Normalizing numeric features...


✅ Saved scaler.pkl


# ------------------------------------------------------------
# 7. DIMENSIONALITY REDUCTION (OPTIONAL PCA)
# ------------------------------------------------------------

In [11]:
print("🔹 Performing PCA on numeric features (for compact embedding)...")
pca = PCA(n_components=3, random_state=42)
pca_features = pca.fit_transform(df[num_cols])
pca_cols = [f"pca_{i+1}" for i in range(pca_features.shape[1])]
df[pca_cols] = pca_features
joblib.dump(pca, "clean_dataset/pca.pkl")
print("✅ Saved pca.pkl")

🔹 Performing PCA on numeric features (for compact embedding)...
✅ Saved pca.pkl


# ------------------------------------------------------------
# 8. IMBALANCED DATA HANDLING (CLASS WEIGHTS)
# ------------------------------------------------------------

In [12]:
print("🔹 Computing class weights...")
classes = np.unique(df["overall"])
class_weights = compute_class_weight("balanced", classes=classes, y=df["overall"])

# Convert numpy int64 to regular Python integers
class_weights_dict = dict(zip(classes.tolist(), class_weights.tolist()))

with open("clean_dataset/class_weights.json", "w") as f:
    json.dump(class_weights_dict, f, indent=2)

print("✅ Saved class_weights.json")
print("Class Weights:", class_weights_dict)

🔹 Computing class weights...


✅ Saved class_weights.json
Class Weights: {1: 2.0197847631786954, 2: 2.951209111423038, 3: 2.0646247933473383, 4: 1.0707231540811986, 5: 0.3639333558285166}


# ------------------------------------------------------------
# 9. DATA SPLITTING (GROUPED BY PRODUCT ASIN)
# ------------------------------------------------------------

In [13]:
# ------------------------------------------------------------
# 7. TRAIN/VALIDATION SPLIT (GROUPED BY ASIN)
# ------------------------------------------------------------
print("🔹 Splitting train/validation sets...")

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["asin"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
print(f"Train shape: {train_df.shape}, Validation shape: {val_df.shape}")

🔹 Splitting train/validation sets...
Train shape: (747564, 26), Validation shape: (82921, 26)


# ------------------------------------------------------------
# 10. FINAL FEATURE SELECTION
# ------------------------------------------------------------

In [14]:
cols_to_keep = [
    "overall",
    "text_clean",
    "verified",
    "vote_count",
    # "asin",
    "review_len_chars",
    "review_len_words",
    "summary_len",
    "asin_freq",
    "style_freq",
    "style_encoded",
    "review_year",
    "review_month",
]

train_clean = train_df[cols_to_keep]
val_clean = val_df[cols_to_keep]

# ------------------------------------------------------------
# 11. EXPORT CLEAN DATASETS
# ------------------------------------------------------------

In [15]:

train_clean.to_csv("clean_dataset/train_clean.csv", index=False)
val_clean.to_csv("clean_dataset/val_clean.csv", index=False)
print("✅ Exported train_clean.csv and val_clean.csv")

# BERT text-only versions
bert_train = train_df[['text_clean', 'overall']]
bert_val = val_df[['text_clean', 'overall']]
bert_train.to_csv("clean_dataset/bert_train.csv", index=False)
bert_val.to_csv("clean_dataset/bert_val.csv", index=False)
print("✅ Exported bert_train.csv and bert_val.csv")


✅ Exported train_clean.csv and val_clean.csv
✅ Exported bert_train.csv and bert_val.csv



# ------------------------------------------------------------
# 12. PIPELINE EXPORTS 
# ------------------------------------------------------------

In [16]:
joblib.dump(
    {
        "scaler": scaler,
        "pca": pca,
        "label_encoder_style": le_style,
        "class_weights": class_weights_dict,
    },
    "clean_dataset/preprocessing_pipeline.pkl",
)

print("✅ Saved preprocessing_pipeline.pkl")

✅ Saved preprocessing_pipeline.pkl


# ------------------------------------------------------------
# 13. SUMMARY
# ------------------------------------------------------------

In [17]:

print("\n🎯 Preprocessing completed successfully!")
print("Generated files:")
print(" - train_clean.csv / val_clean.csv → Full numeric + text dataset")
print(" - bert_train.csv / bert_val.csv → Text-only datasets for BERT")
print(" - scaler.pkl / pca.pkl / preprocessing_pipeline.pkl")
print(" - class_weights.json → For model training balancing")


🎯 Preprocessing completed successfully!
Generated files:
 - train_clean.csv / val_clean.csv → Full numeric + text dataset
 - bert_train.csv / bert_val.csv → Text-only datasets for BERT
 - scaler.pkl / pca.pkl / preprocessing_pipeline.pkl
 - class_weights.json → For model training balancing
